In [ ]:
"""
Multi-View Dataset Curation & Clinical Text Preprocessing
----------------------------------------------------------------
Pairs Frontal and Lateral chest X-ray projections per study (uid) from the
Indiana University Chest X-Ray (IU X-Ray) dataset, cleans clinical text fields
(findings and indication), removes empty records, and performs an 80:10:10
train/validation/test split.
"""

In [ ]:
import numpy as np
import pandas as pd
df_reports= pd.read_csv('/kaggle/input/datasets/raddar/chest-xrays-indiana-university/indiana_reports.csv')
df_reports.head()

,uid,MeSH,Problems,image,indication,comparison,findings,impression
0,1,normal,normal,Xray Chest PA and Lateral,Positive TB test,None.,The cardiac silhouette and mediastinum size ar...,Normal chest x-XXXX.
1,2,Cardiomegaly/borderline;Pulmonary Artery/enlarged,Cardiomegaly;Pulmonary Artery,"Chest, 2 views, frontal and lateral",Preop bariatric surgery.,None.,Borderline cardiomegaly. Midline sternotomy XX...,No acute pulmonary findings.
2,3,normal,normal,Xray Chest PA and Lateral,"rib pain after a XXXX, XXXX XXXX steps this XX...",NaN,NaN,"No displaced rib fractures, pneumothorax, or p..."
3,4,"Pulmonary Disease, Chronic Obstructive;Bullous...","Pulmonary Disease, Chronic Obstructive;Bullous...","PA and lateral views of the chest XXXX, XXXX a...",XXXX-year-old XXXX with XXXX.,None available,There are diffuse bilateral interstitial and a...,1. Bullous emphysema and interstitial fibrosis...
4,5,Osteophyte/thoracic vertebrae/multiple/small;T...,Osteophyte;Thickening;Lung,Xray Chest PA and Lateral,Chest and nasal congestion.,NaN,The cardiomediastinal silhouette and pulmonary...,No acute cardiopulmonary abnormality.


In [3]:
df_projection= pd.read_csv('/kaggle/input/datasets/raddar/chest-xrays-indiana-university/indiana_projections.csv')
df_projection.head()

,uid,filename,projection
0,1,1_IM-0001-4001.dcm.png,Frontal
1,1,1_IM-0001-3001.dcm.png,Lateral
2,2,2_IM-0652-1001.dcm.png,Frontal
3,2,2_IM-0652-2001.dcm.png,Lateral
4,3,3_IM-1384-1001.dcm.png,Frontal


In [4]:
df_reports.isnull().sum()

uid              0
MeSH             0
Problems         0
image            0
indication      86
comparison    1166
findings       514
impression      31
dtype: int64

In [5]:
df_projection.isnull().sum()

uid           0
filename      0
projection    0
dtype: int64

In [6]:
df_projection.shape

(7466, 3)

In [ ]:

PROJECTIONS_CSV = "/kaggle/input/datasets/raddar/chest-xrays-indiana-university/indiana_projections.csv"   # has: uid, filename, projection
REPORTS_CSV = "/kaggle/input/datasets/raddar/chest-xrays-indiana-university/indiana_reports.csv"           # has: uid, findings, indication, ...
OUTPUT_CSV = "/kaggle/working/merged_dataset.csv"


IMAGE_DIR = "/kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_normalized/"

def build_path(filename: str) -> str:
    if not IMAGE_DIR:
        return filename
    return IMAGE_DIR.rstrip("/") + "/" + filename


def main():
    proj_df = pd.read_csv(PROJECTIONS_CSV)
    rep_df = pd.read_csv(REPORTS_CSV)

    # Normalize projection labels (in case of casing/whitespace issues)
    proj_df["projection"] = proj_df["projection"].str.strip().str.capitalize()

    # Keep only Frontal/Lateral rows
    proj_df = proj_df[proj_df["projection"].isin(["Frontal", "Lateral"])]

    # Pivot so each uid gets one row with separate frontal/lateral filename columns
    pivot_df = proj_df.pivot_table(
        index="uid",
        columns="projection",
        values="filename",
        aggfunc="first"  # in case of duplicates, take the first
    ).reset_index()

    # Ensure both columns exist even if some uid never had that projection
    for col in ["Frontal", "Lateral"]:
        if col not in pivot_df.columns:
            pivot_df[col] = pd.NA

    # Drop patients missing either frontal or lateral image
    pivot_df = pivot_df.dropna(subset=["Frontal", "Lateral"])

    # Build the combined "path" column: frontal path, lateral path
    pivot_df["path"] = pivot_df.apply(
        lambda row: f"{build_path(row['Frontal'])},{build_path(row['Lateral'])}",
        axis=1
    )

    # Merge with reports on uid to bring in findings & indication
    merged = pd.merge(
        pivot_df[["uid", "path"]],
        rep_df[["uid", "findings", "indication"]],
        on="uid",
        how="inner"   # only keep uids present in both files
    )

    # Reorder / select final columns
    final_df = merged[["uid", "path", "findings", "indication"]]

    final_df.to_csv(OUTPUT_CSV, index=False)
    print(f"Done. Wrote {len(final_df)} rows to {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

Done. Wrote 3388 rows to /kaggle/working/merged_dataset.csv


In [2]:
df_merge=pd.read_csv('/kaggle/working/merged_dataset.csv')

In [3]:
df_merge

,uid,path,findings,indication
0,1,/kaggle/input/datasets/raddar/chest-xrays-indi...,The cardiac silhouette and mediastinum size ar...,Positive TB test
1,2,/kaggle/input/datasets/raddar/chest-xrays-indi...,Borderline cardiomegaly. Midline sternotomy XX...,Preop bariatric surgery.
2,3,/kaggle/input/datasets/raddar/chest-xrays-indi...,NaN,"rib pain after a XXXX, XXXX XXXX steps this XX..."
3,4,/kaggle/input/datasets/raddar/chest-xrays-indi...,There are diffuse bilateral interstitial and a...,XXXX-year-old XXXX with XXXX.
4,5,/kaggle/input/datasets/raddar/chest-xrays-indi...,The cardiomediastinal silhouette and pulmonary...,Chest and nasal congestion.
...,...,...,...,...
3383,3995,/kaggle/input/datasets/raddar/chest-xrays-indi...,The cardiomediastinal silhouette and pulmonary...,"Nausea, vomiting x2 weeks. Dialysis patient."
3384,3996,/kaggle/input/datasets/raddar/chest-xrays-indi...,The lungs are clear. Heart size is normal. No ...,NaN
3385,3997,/kaggle/input/datasets/raddar/chest-xrays-indi...,"Heart size within normal limits. Small, nodula...",XXXX-year-old male with positive PPD.
3386,3998,/kaggle/input/datasets/raddar/chest-xrays-indi...,NaN,tuberculosis positive PPD


In [6]:
df_merge.shape

(3388, 4)

In [7]:
df_merge.dropna(subset={'findings'},inplace=True)

In [8]:
df_merge.shape

(2943, 4)

In [9]:
df_merge.isnull().sum()

uid            0
path           0
findings       0
indication    47
dtype: int64

In [10]:
df_merge.dropna(subset={'indication'},inplace=True)

In [11]:
df_merge.shape

(2896, 4)

In [12]:
df_merge.isnull().sum()

uid           0
path          0
findings      0
indication    0
dtype: int64

In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split

# ---- Settings ----
TRAIN_CSV = "/kaggle/working/train.csv"
VAL_CSV = "/kaggle/working/val.csv"
TEST_CSV = "/kaggle/working/test.csv"

TRAIN_SIZE = 0.8   # 80% train
VAL_SIZE = 0.1      # 10% validation
TEST_SIZE = 0.1      # 10% test
RANDOM_STATE = 42


# ---- First split: train vs (val+test) ----
train_df, temp_df = train_test_split(
    df_merge,
    train_size=TRAIN_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True
)

# ---- Second split: val vs test (split the remaining part) ----
relative_test_size = TEST_SIZE / (VAL_SIZE + TEST_SIZE)
val_df, test_df = train_test_split(
    temp_df,
    test_size=relative_test_size,
    random_state=RANDOM_STATE,
    shuffle=True
)

# ---- Save ----
train_df.to_csv(TRAIN_CSV, index=False)
val_df.to_csv(VAL_CSV, index=False)
test_df.to_csv(TEST_CSV, index=False)

print(f"Train: {len(train_df)} rows")
print(f"Val:   {len(val_df)} rows")
print(f"Test:  {len(test_df)} rows")

Train: 2316 rows
Val:   290 rows
Test:  290 rows


In [ ]:
import re

COLUMNS_TO_CLEAN = ["findings", "indication"]  # Columns to preprocess

def clean_text(text):
    """Standard clinical text cleaning as per the research paper."""
    if not isinstance(text, str):
        return ""
    # Lowercase
    text = text.lower()
    # Remove the 'XXXX' placeholders often found in de-identified datasets
    text = text.replace('xxxx', '')
    # Remove multiple spaces and newlines
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def preprocess_dataframe(df, columns, split_name):
    """Apply clean_text to specified columns of a dataframe."""
    
    print(f"\n{split_name} DataFrame:")
    print(f"  Records: {len(df)}")
    
    for col in columns:
        if col in df.columns:
            df[col] = df[col].apply(clean_text)
            print(f"Cleaned column: '{col}'")
        else:
            print(f" Column '{col}' not found in {split_name} — skipped")
    
    return df

# Preprocess all splits
train_df = preprocess_dataframe(train_df, COLUMNS_TO_CLEAN, "Train")
test_df  = preprocess_dataframe(test_df,  COLUMNS_TO_CLEAN, "Test")
val_df   = preprocess_dataframe(val_df,   COLUMNS_TO_CLEAN, "Validation")

# Preview cleaned samples
print("\nSample cleaned values from Train:")
for col in COLUMNS_TO_CLEAN:
    if col in train_df.columns:
        print(f"\n  [{col}]:")
        print(f"  {train_df[col].iloc[0]}")

print("\nPreprocessing complete!")


Train DataFrame:
  Records: 2316
Cleaned column: 'findings'
Cleaned column: 'indication'

Test DataFrame:
  Records: 290
Cleaned column: 'findings'
Cleaned column: 'indication'

Validation DataFrame:
  Records: 290
Cleaned column: 'findings'
Cleaned column: 'indication'

Sample cleaned values from Train:

  [findings]:
  heart is enlarged. there is prominence of the central pulmonary vasculature. mild diffuse interstitial opacities bilaterally, predominantly in the bases, with no focal consolidation, pleural effusion, or pneumothoraces. and soft tissues are unremarkable.

  [indication]:
  -year-old male with dyspnea, chest pain

Preprocessing complete!


In [20]:
train_df.isnull().sum()

uid           0
path          0
findings      0
indication    0
dtype: int64

In [21]:
print(train_df['indication'].str.strip().eq('').sum())
print(val_df['indication'].str.strip().eq('').sum())
print(test_df['indication'].str.strip().eq('').sum())

74
9
15


In [22]:
print(train_df['findings'].str.strip().eq('').sum())
print(val_df['findings'].str.strip().eq('').sum())
print(test_df['findings'].str.strip().eq('').sum())

0
0
0


In [23]:
import numpy as np

# Step 1: Convert all blank/whitespace-only strings to actual NaN (Not a Number) values
train_df['indication'] = train_df['indication'].replace(r'^\s*$', np.nan, regex=True)

# Step 2: Drop all rows where 'indication' is NaN
train_df = train_df.dropna(subset=['indication']).reset_index(drop=True)



# Step 1: Convert all blank/whitespace-only strings to actual NaN (Not a Number) values
val_df['indication'] = val_df['indication'].replace(r'^\s*$', np.nan, regex=True)

# Step 2: Drop all rows where 'indication' is NaN
val_df = val_df.dropna(subset=['indication']).reset_index(drop=True)



# Step 1: Convert all blank/whitespace-only strings to actual NaN (Not a Number) values
test_df['indication'] = test_df['indication'].replace(r'^\s*$', np.nan, regex=True)

# Step 2: Drop all rows where 'indication' is NaN
test_df = test_df.dropna(subset=['indication']).reset_index(drop=True)

In [24]:
print(train_df['indication'].str.strip().eq('').sum())
print(val_df['indication'].str.strip().eq('').sum())
print(test_df['indication'].str.strip().eq('').sum())

0
0
0


In [25]:
print(train_df['findings'].str.strip().eq('').sum())
print(val_df['findings'].str.strip().eq('').sum())
print(test_df['findings'].str.strip().eq('').sum())

0
0
0


In [26]:
print(train_df.isnull().sum())
print(val_df.isnull().sum())
print(test_df.isnull().sum())

uid           0
path          0
findings      0
indication    0
dtype: int64
uid           0
path          0
findings      0
indication    0
dtype: int64
uid           0
path          0
findings      0
indication    0
dtype: int64


In [27]:
train_df.to_csv("Multimodal_train_multiview_final.csv", index=False)
val_df.to_csv("Multimodal_val_multiview_final.csv", index=False)
test_df.to_csv("Multimodal_test_multiview_final.csv", index=False)